# VisionGuard Medical — Kaggle Stage A Preflight

**Preflight only. No authentication, no downloads, no large installs.**

This notebook verifies the Kaggle environment is suitable for a later Stage A run against RadGenome-ChestCT.
It does NOT:

- authenticate to Hugging Face or request credentials,
- download any CT volume, mask, or large archive,
- install ML libraries (torch, nibabel, etc.),
- upload any data back out of Kaggle.

**Redistribution reminder**: RadGenome-ChestCT and its CT-RATE source forbid redistribution. Do not save CT volumes, masks, patient reports, or any case-identifying derivative as Kaggle Dataset or notebook outputs. Keep this notebook's output limited to the preflight numbers below.

## 1. Python / platform / interpreter

In [ ]:
import sys, platform
print('python:', sys.version.split()[0])
print('executable:', sys.executable)
print('platform:', platform.platform())
print('machine:', platform.machine())

## 2. Disk — working dir + ephemeral scratch

In [ ]:
import shutil, os
GB = 1024 ** 3
for path in ['/kaggle/working', '/kaggle/temp', '/tmp', '/']:
    if os.path.isdir(path):
        total, used, free = shutil.disk_usage(path)
        print(f'{path:16s}  total={total/GB:7.2f} GiB  free={free/GB:7.2f} GiB')
    else:
        print(f'{path:16s}  (not present)')

## 3. RAM

In [ ]:
GiB = 1024 ** 3
info = {}
try:
    with open('/proc/meminfo') as f:
        for line in f:
            k, _, rest = line.partition(':')
            info[k.strip()] = rest.strip()
    for k in ('MemTotal', 'MemAvailable', 'MemFree', 'SwapTotal'):
        if k in info:
            kb = int(info[k].split()[0])
            print(f'{k:14s}  {kb*1024/GiB:7.2f} GiB')
except FileNotFoundError:
    print('/proc/meminfo not available on this platform')

## 4. GPU — info only, no CUDA kernels run

If the notebook was started with a GPU accelerator attached, `nvidia-smi` will report it. We do not import torch here (would waste time).

In [ ]:
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                         capture_output=True, text=True, timeout=10)
    print(out.stdout.strip() or '(no GPU rows)')
    if out.stderr.strip():
        print('STDERR:', out.stderr.strip())
except FileNotFoundError:
    print('nvidia-smi not installed — no GPU attached to this session')

## 5. Hugging Face Hub client — import only, NO login, NO download

`huggingface_hub` is pre-installed on Kaggle images. We verify it imports and we check the version. We do NOT call `login()`, `whoami()`, or any file download here.

In [ ]:
try:
    import huggingface_hub
    print('huggingface_hub:', huggingface_hub.__version__)
    # Confirm the two symbols Stage A will need are importable (no network call).
    from huggingface_hub import HfApi, hf_hub_download  # noqa: F401
    print('HfApi + hf_hub_download: importable')
except ImportError as exc:
    print('huggingface_hub NOT available:', exc)
    print('If missing: `pip install "huggingface_hub"` is small (<5 MB) and allowed by the no-large-installs rule.')

## 6. Kaggle Secrets — check WHETHER `HF_TOKEN` is configured, but do NOT read it

If you have added an `HF_TOKEN` secret in the notebook's **Add-ons → Secrets** panel, Kaggle exposes it to `UserSecretsClient`. This cell checks that the secret *exists* without printing its value. If the lookup raises, the secret is not attached — add it before Stage A.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    us = UserSecretsClient()
    try:
        token = us.get_secret('HF_TOKEN')
        # Do NOT print the token. Report length only, as evidence it was retrieved.
        print(f"HF_TOKEN secret: present (length={len(token)} chars) — value NOT printed")
        del token
    except Exception as exc:  # noqa: BLE001
        print(f'HF_TOKEN secret: NOT configured ({type(exc).__name__})')
        print('To add: Add-ons → Secrets → Add secret with label HF_TOKEN.')
except ImportError:
    print('kaggle_secrets module not available — this notebook is probably not running on Kaggle.')

## 7. Preflight summary — fill in before requesting Stage A approval

| Check | Value | OK? |
|---|---|---|
| Python 3.10–3.12 | *(from §1)* | |
| `/kaggle/working` free ≥ 15 GiB | *(from §2)* | |
| `/kaggle/temp` or `/tmp` free ≥ 15 GiB | *(from §2)* | |
| RAM ≥ 8 GiB | *(from §3)* | |
| GPU | *(from §4 — optional for Stage A)* | n/a |
| `huggingface_hub` importable | *(from §5)* | |
| `HF_TOKEN` secret configured | *(from §6)* | |
| HF gate form accepted on dataset page | *(verify in browser)* | |
| Notebook is set to **not** save outputs beyond preflight text | *(Settings → Save Output)* | |

**Do not proceed to Stage A until every row above is ✅ and the human partner has given explicit approval.**

## 8. What Stage A will do (for context — not executed here)

Once approved, Stage A will:

1. Authenticate HF via the `HF_TOKEN` secret (`huggingface_hub.login(token=...)`).
2. Download — in order — `validation_region_report.csv` (9.17 MiB), `report_hierarchy_tree.zip` (97.5 MiB), `valid_1_a_1.nii.gz` (25.6 MiB), `valid_anatomy_mask.tar.gz` (9.78 GiB).
3. Extract **only** the mask entry matching `valid_1_a_1` from the tarball and delete the tarball.
4. Run the alignment checks from `docs/sample_download_plan.md §8`.
5. Emit **only** a small text/JSON report (shapes, affines, label histogram, mapping discovery) back to the notebook output — **no CT volumes, masks, or report text leave the session**.

None of that happens in this preflight.